# Phase 5 Stage 1.5 — Walk-forward backtest results

Loads all 72 Stage 1.5 audit logs + summaries from `data/backtests/stage15/`. Surfaces:
- master comparison table (PRIMARY + SIDEBAR splits)
- Sharpe heatmap by cohort × bucket × sizing × window
- top combinations by Sharpe / PnL / joint metric
- cross-window robustness
- **slippage distribution analysis** (new in Stage 1.5)
- **fallback-rate analysis by market liquidity**
- **per-leader / per-rank signal frequency**
- **diff vs Stage 1** with audit-log-level comparison
- Stage 2 candidate selection

**Stage 1.5 differs from Stage 1 in three compound ways**:
1. TopK=10 cohort selection (vs all-qualifiers)
2. 1% sizing + tighter caps (vs 2%)
3. **Next-fill slippage model** (vs Stage 1's constant 2/4/2¢)

Diff results below shouldn't be read as pure strategy-shape effect — slippage model is also changing.

## Setup

In [ ]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STAGE15_DIR = ROOT / 'data' / 'backtests' / 'stage15'
STAGE1_DIR = ROOT / 'data' / 'backtests'
WINDOWS_PRIMARY = {'2025', '2026'}
STARTING_CAPITAL = 100_000.0  # strategy_capital_usd default in walkforward_stage15.py
# Window length in months for annualization
WINDOW_MONTHS = {'2024': 12.0, '2025': 12.0, '2026': 3.77}  # 2026 = Jan 1 to Apr 23

def flatten_15(s):
    window = s['test_window_start'][:4]
    pnl = s['total_pnl_usd']
    ret = pnl / STARTING_CAPITAL
    months = WINDOW_MONTHS.get(window, 12.0)
    return {
        'cohort': s['cohort'], 'bucket': s['resolution_bucket'],
        'sizing': s['sizing_rule'], 'window': window,
        'is_primary': window in WINDOWS_PRIMARY,
        'n_signals': s['n_signals'], 'total_pnl': pnl,
        'ret_pct': ret, 'ret_pct_annual': ret * (12.0 / months),
        'sharpe': s['sharpe_monthly'], 'sortino': s.get('sortino_monthly', 0),
        'win_rate': s.get('win_rate') or 0, 'profit_factor': s.get('profit_factor') or 0,
        'deploy_ratio': s['deployment_ratio'], 'sig_per_wk': s['signals_per_week'],
        'max_dd': s.get('max_drawdown_usd', 0),
        'max_dd_pct': s.get('max_drawdown_usd', 0) / STARTING_CAPITAL,
        'fallback_pct': s.get('slippage_fallback_pct', 0),
        'avg_slip_c': s.get('slippage_avg_cents', 0),
        'p50_slip_c': s.get('slippage_p50_cents', 0),
        'avg_size': s.get('avg_signal_size_usd', 0),
        'n_leaders': s.get('n_distinct_leaders', 0),
        'n_markets': s.get('n_distinct_markets', 0),
        'negrisk_share': s.get('negrisk_signal_share', 0),
    }

summaries = [json.loads(p.read_text()) for p in sorted(STAGE15_DIR.glob('*_summary.json'))]
df = pd.DataFrame([flatten_15(s) for s in summaries])
print(f'loaded {len(df)} Stage 1.5 summaries')
print(f'PnL displayed as % of ${STARTING_CAPITAL:,.0f} starting capital')
print(f'  ret_pct        = window total return on $100k')
print(f'  ret_pct_annual = annualized (×12/window_months)')
df.head(3)

## Master comparison table — PRIMARY windows (2025, 2026)

In [ ]:
primary = df[df['is_primary']].copy()
cols = ['cohort','bucket','sizing','window','n_signals','ret_pct','ret_pct_annual','sharpe','deploy_ratio',
        'fallback_pct','avg_slip_c','avg_size','n_leaders']
primary[cols].sort_values(['cohort','bucket','sizing','window']).style.format({
    'sharpe': '{:.2f}',
    'ret_pct': '{:+.2%}', 'ret_pct_annual': '{:+.2%}',
    'deploy_ratio': '{:.0%}', 'fallback_pct': '{:.0%}',
    'avg_slip_c': '{:.2f}', 'avg_size': '${:,.0f}',
})

## Sidebar table — 2024 (low confidence, data-sparse era)

In [ ]:
sb = df[~df['is_primary']].copy()
sb[cols].sort_values(['cohort','bucket','sizing']).style.format({
    'sharpe': '{:.2f}',
    'ret_pct': '{:+.2%}', 'ret_pct_annual': '{:+.2%}',
    'deploy_ratio': '{:.0%}', 'fallback_pct': '{:.0%}',
    'avg_slip_c': '{:.2f}', 'avg_size': '${:,.0f}',
})

## Sharpe heatmap — cohort × bucket × sizing × window

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 10), squeeze=False)
for i, window in enumerate(sorted(df['window'].unique())):
    for j, sizing in enumerate(sorted(df['sizing'].unique())):
        sub = df[(df['window']==window) & (df['sizing']==sizing)]
        pivot = sub.pivot(index='cohort', columns='bucket', values='sharpe')
        pivot = pivot[['2d','7d','30d','60d']]
        ax = axes[i, j]
        im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-4, vmax=4, aspect='auto')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
        for r in range(pivot.shape[0]):
            for c in range(pivot.shape[1]):
                v = pivot.iat[r, c]
                if pd.notna(v):
                    ax.text(c, r, f'{v:.1f}', ha='center', va='center',
                            color='black' if abs(v) < 2 else 'white', fontsize=10)
        sidebar_tag = ' (SIDEBAR)' if window not in WINDOWS_PRIMARY else ''
        ax.set_title(f'{window}{sidebar_tag} — {sizing}', fontsize=10)
fig.suptitle('Sharpe — Stage 1.5', y=1.0)
fig.colorbar(im, ax=axes.ravel().tolist(), label='Sharpe (monthly × √12)')
plt.tight_layout()

## Top combinations — three ranking views

PRIMARY windows only. Same configuration may appear once per window.

In [ ]:
primary['joint'] = primary['sharpe'] * primary['deploy_ratio']
print('=== TOP 10 BY SHARPE ===')
display(primary.sort_values('sharpe', ascending=False).head(10)[
    ['cohort','bucket','sizing','window','sharpe','ret_pct','ret_pct_annual','deploy_ratio','n_signals','fallback_pct','avg_slip_c']
].style.format({
    'sharpe': '{:.2f}', 'ret_pct': '{:+.2%}', 'ret_pct_annual': '{:+.2%}',
    'deploy_ratio': '{:.0%}', 'fallback_pct': '{:.0%}', 'avg_slip_c': '{:.2f}',
}))

In [ ]:
print('=== TOP 10 BY RETURN (annualized) ===')
display(primary.sort_values('ret_pct_annual', ascending=False).head(10)[
    ['cohort','bucket','sizing','window','ret_pct','ret_pct_annual','sharpe','deploy_ratio','n_signals','fallback_pct']
].style.format({
    'sharpe': '{:.2f}', 'ret_pct': '{:+.2%}', 'ret_pct_annual': '{:+.2%}',
    'deploy_ratio': '{:.0%}', 'fallback_pct': '{:.0%}',
}))

In [ ]:
print('=== TOP 10 BY SHARPE × DEPLOY ===')
display(primary.sort_values('joint', ascending=False).head(10)[
    ['cohort','bucket','sizing','window','sharpe','deploy_ratio','joint','ret_pct','ret_pct_annual','n_signals']
].style.format({
    'sharpe': '{:.2f}', 'deploy_ratio': '{:.0%}', 'joint': '{:.3f}',
    'ret_pct': '{:+.2%}', 'ret_pct_annual': '{:+.2%}',
}))

## Cross-window robustness — PRIMARY windows only

Number of windows with positive Sharpe is the robustness signal. Mean Sharpe across primary windows is the headline. Only configurations where the strategy is positive on BOTH 2025 AND 2026 are real candidates.

In [ ]:
rob = primary.groupby(['cohort','bucket','sizing']).agg(
    sharpe_mean=('sharpe','mean'), sharpe_min=('sharpe','min'), sharpe_max=('sharpe','max'),
    ret_pct_total=('ret_pct','sum'),
    ret_pct_annual_mean=('ret_pct_annual','mean'),
    n_pos_windows=('sharpe', lambda x: (x>0).sum()),
    avg_deploy=('deploy_ratio','mean'), avg_n_signals=('n_signals','mean'),
    avg_fallback=('fallback_pct','mean'),
).reset_index()
rob['joint'] = rob['sharpe_mean'] * rob['avg_deploy']
rob.sort_values(['n_pos_windows','joint'], ascending=[False, False]).style.format({
    'sharpe_mean':'{:.2f}','sharpe_min':'{:.2f}','sharpe_max':'{:.2f}',
    'ret_pct_total':'{:+.2%}','ret_pct_annual_mean':'{:+.2%}',
    'avg_deploy':'{:.0%}','avg_n_signals':'{:.0f}',
    'avg_fallback':'{:.0%}','joint':'{:.3f}',
})

## Slippage distribution analysis (new in Stage 1.5)

The next-fill model produces a *distribution* of per-trade slippages — unlike Stage 1's constant cents. Negative slippage = market moved in our favour between leader fill and next fill. Heavy right tail = fallback dominance.

In [ ]:
con = duckdb.connect()
audits_primary = sorted(str(p) for p in STAGE15_DIR.glob('*.parquet')
                        if p.name.split('__')[3] in WINDOWS_PRIMARY)
audits_sql = ', '.join(f"'{p}'" for p in audits_primary)
all_audits = con.sql(f"SELECT * FROM read_parquet([{audits_sql}])").fetchdf()
print(f'loaded {len(all_audits):,} PRIMARY audit rows from {len(audits_primary)} runs')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cohort in zip(axes, sorted(all_audits['cohort'].unique())):
    sub = all_audits[all_audits['cohort']==cohort]
    next_fill = sub[sub['slippage_source']=='next_fill']['slippage_cents']
    fallback = sub[sub['slippage_source']=='fallback']['slippage_cents']
    bins = np.linspace(0, 6, 40)
    ax.hist(next_fill, bins=bins, alpha=0.7, label=f'next_fill ({len(next_fill):,})', color='steelblue')
    ax.hist(fallback, bins=bins, alpha=0.7, label=f'fallback ({len(fallback):,})', color='salmon')
    ax.axvline(3.0, color='red', ls='--', alpha=0.5, label='fallback default 3¢')
    ax.set_xlabel('slippage_cents')
    ax.set_ylabel('count')
    ax.set_title(f'{cohort}\navg={sub["slippage_cents"].mean():.2f}¢, '
                 f'fallback={len(fallback)/len(sub):.0%}')
    ax.legend(fontsize=8)
fig.suptitle('Slippage distribution per cohort (PRIMARY windows)', y=1.02)
plt.tight_layout()

In [ ]:
# Per-cohort SQL the user asked for (pnl as % of $100k starting capital)
con.register('all_audits', all_audits)
con.sql(f"""
    SELECT cohort,
           COUNT(*) AS n_signals,
           ROUND(AVG(slippage_cents), 3) AS avg_slip_c,
           ROUND(STDDEV(slippage_cents), 3) AS slip_std,
           ROUND(SUM(CASE WHEN slippage_source='fallback' THEN 1 ELSE 0 END)::DOUBLE / COUNT(*), 4) AS fallback_pct,
           ROUND(AVG(copy_size_usd), 2) AS avg_size,
           ROUND(SUM(copy_pnl_usd) / {STARTING_CAPITAL} * 100, 2) AS ret_pct,
           ROUND(SUM(copy_pnl_usd) FILTER (WHERE slippage_source='next_fill') / {STARTING_CAPITAL} * 100, 2) AS ret_pct_next_fill,
           ROUND(SUM(copy_pnl_usd) FILTER (WHERE slippage_source='fallback') / {STARTING_CAPITAL} * 100, 2) AS ret_pct_fallback
    FROM all_audits
    GROUP BY cohort
""").fetchdf()

## Fallback rate vs market liquidity

Stage 1.5 filtered to `market_volume > $50k` but markets still span 50k → millions. Hypothesis: fallback rate (no qualifying next-fill within 15-300s) is higher on smaller markets. If yes, the slippage model is least reliable exactly where the market filter is weakest.

In [ ]:
all_audits['vol_bucket'] = pd.cut(all_audits['market_volume'],
    bins=[5e4, 1e5, 5e5, 1e6, 5e6, 1e7, 1e8, 1e10],
    labels=['50k-100k','100k-500k','500k-1M','1M-5M','5M-10M','10M-100M','>100M'])
fb_by_vol = (all_audits.groupby(['vol_bucket','cohort'], observed=True)
              .agg(n=('slippage_source','count'),
                   fallback_pct=('slippage_source', lambda x: (x=='fallback').mean()),
                   ret_pct=('copy_pnl_usd', lambda x: x.sum() / STARTING_CAPITAL))
              .reset_index())
fb_by_vol.pivot(index='vol_bucket', columns='cohort', values='fallback_pct').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for cohort in sorted(fb_by_vol['cohort'].unique()):
    sub = fb_by_vol[fb_by_vol['cohort']==cohort].sort_values('vol_bucket')
    ax.plot(sub['vol_bucket'].astype(str), sub['fallback_pct'], marker='o', label=cohort, lw=2)
ax.set_xlabel('market lifetime volume bucket')
ax.set_ylabel('fallback_pct')
ax.set_title('Fallback rate vs market liquidity — higher liquidity, less fallback')
ax.axhline(0.1, color='gray', ls='--', alpha=0.5, label='10% fallback floor')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## PnL by slippage source

Does the fallback (3¢ adverse hard-coded) account for an outsized share of losses? If yes, raising the market liquidity threshold to suppress fallbacks could improve PnL even at the cost of fewer signals.

In [ ]:
pnl_by_src = (all_audits.groupby(['cohort','slippage_source'])
              .agg(n=('copy_pnl_usd','count'),
                   ret_pct=('copy_pnl_usd', lambda x: x.sum() / STARTING_CAPITAL),
                   mean_pnl_bps=('copy_pnl_usd', lambda x: x.mean() / STARTING_CAPITAL * 10000),
                   win_rate=('copy_pnl_usd', lambda x: (x>0).mean()))
              .reset_index())
pnl_by_src.style.format({
    'ret_pct': '{:+.2%}',
    'mean_pnl_bps': '{:+.2f} bps',
    'win_rate': '{:.2%}',
})

## Per-leader-rank signal frequency

TopK=10 ranks leaders 1-10. Rank 1 = highest mkt_total_pnl × profit_factor. Do higher-ranked leaders contribute more PnL? Or are they unpredictably distributed?

In [ ]:
by_rank = (all_audits.groupby(['cohort','selected_leader_rank'])
           .agg(n=('copy_pnl_usd','count'),
                ret_pct=('copy_pnl_usd', lambda x: x.sum() / STARTING_CAPITAL),
                mean_pnl=('copy_pnl_usd','mean'))
           .reset_index())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cohort in zip(axes, sorted(by_rank['cohort'].unique())):
    sub = by_rank[by_rank['cohort']==cohort].sort_values('selected_leader_rank')
    color = ['steelblue' if p >= 0 else 'salmon' for p in sub['ret_pct']]
    ax.bar(sub['selected_leader_rank'], sub['ret_pct'] * 100, color=color)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xlabel('selected_leader_rank (1 = top, 10 = bottom)')
    ax.set_ylabel('return contribution (% of $100k)')
    ax.set_title(f'{cohort}')
fig.suptitle('Return contribution by leader-rank — PRIMARY windows', y=1.02)
plt.tight_layout()

In [ ]:
# Per-leader concentration: how many leaders carry the return?
by_leader = (all_audits.groupby(['cohort','leader_address'])
             .agg(n_signals=('copy_pnl_usd','count'),
                  ret_pct=('copy_pnl_usd', lambda x: x.sum() / STARTING_CAPITAL))
             .reset_index())
print('Distinct leaders per cohort (PRIMARY windows):')
print(by_leader.groupby('cohort')['leader_address'].count())
print('\nTop 5 leaders by absolute return contribution per cohort:')
for cohort in sorted(by_leader['cohort'].unique()):
    sub = by_leader[by_leader['cohort']==cohort].copy()
    sub['abs_ret'] = sub['ret_pct'].abs()
    print(f'\n{cohort}:')
    display(sub.sort_values('abs_ret', ascending=False).head(5)[['leader_address','n_signals','ret_pct']]
              .style.format({'ret_pct':'{:+.3%}'}))

## Diff vs Stage 1

Matched configurations (cohort × bucket × sizing × window). Caveat: diff compounds (a) TopK selection, (b) 1% sizing, (c) next-fill slippage model — cannot attribute to any single factor.

In [ ]:
def flatten_1(s):
    return {'cohort': s['cohort'], 'bucket': s['resolution_bucket'],
            'sizing': s['sizing_rule'], 'window': s['test_window_start'][:4],
            'n_signals_s1': s['n_signals'],
            'ret_pct_s1': s['total_pnl_usd'] / STARTING_CAPITAL,
            'sharpe_s1': s['sharpe_monthly'], 'deploy_ratio_s1': s['deployment_ratio']}

s1 = [json.loads(p.read_text()) for p in sorted(STAGE1_DIR.glob('*_summary.json'))
      if 'stage15' not in p.name]
df1 = pd.DataFrame([flatten_1(s) for s in s1])
merged = df.merge(df1, on=['cohort','bucket','sizing','window'], how='left')
merged['n_sig_reduction_x'] = merged['n_signals_s1'] / merged['n_signals'].clip(1, None)
merged['sharpe_shift'] = merged['sharpe'] - merged['sharpe_s1']
merged['ret_shift_pct'] = merged['ret_pct'] - merged['ret_pct_s1']

primary_diff = merged[merged['is_primary']].sort_values(['cohort','bucket','sizing','window'])
primary_diff[['cohort','bucket','sizing','window',
              'n_signals_s1','n_signals','n_sig_reduction_x',
              'sharpe_s1','sharpe','sharpe_shift',
              'ret_pct_s1','ret_pct','ret_shift_pct']].style.format({
    'n_sig_reduction_x':'{:.1f}x','sharpe_s1':'{:.2f}','sharpe':'{:.2f}',
    'sharpe_shift':'{:+.2f}',
    'ret_pct_s1':'{:+.2%}','ret_pct':'{:+.2%}','ret_shift_pct':'{:+.2%}',
    'n_signals_s1':'{:,}','n_signals':'{:,}',
})

In [ ]:
# Sharpe shift scatter — direction and magnitude
fig, ax = plt.subplots(figsize=(11, 7))
for cohort in sorted(primary_diff['cohort'].unique()):
    sub = primary_diff[primary_diff['cohort']==cohort]
    ax.scatter(sub['sharpe_s1'], sub['sharpe'], s=80, alpha=0.7, label=cohort)
ax.plot([-14, 5], [-14, 5], 'k--', alpha=0.3, lw=1, label='no change')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('Stage 1 Sharpe')
ax.set_ylabel('Stage 1.5 Sharpe')
ax.set_title('Sharpe shift from Stage 1 → Stage 1.5 (above diagonal = improvement)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## Signal timing within OOS windows

Are signals clumped near refresh dates (just-qualified leaders fire fast), or evenly spread? Clumping = front-loading risk. The per-leader monthly cap limits this in theory.

In [ ]:
all_audits['days_since_refresh'] = (
    pd.to_datetime(all_audits['trade_timestamp']) - pd.to_datetime(all_audits['refresh_date'])
).dt.days
fig, ax = plt.subplots(figsize=(11, 4))
for cohort in sorted(all_audits['cohort'].unique()):
    sub = all_audits[all_audits['cohort']==cohort]
    ax.hist(sub['days_since_refresh'], bins=range(0, 32), alpha=0.5, label=cohort)
ax.set_xlabel('days since refresh')
ax.set_ylabel('signal count')
ax.set_title('Signal timing within OOS window (PRIMARY)')
ax.legend()
plt.tight_layout()

## Stage 2 candidate selection

Criteria: positive Sharpe in both 2025 AND 2026; or top-3 by joint metric; deduplicated to 5 final picks. These are what advance to Stage 2 slippage sensitivity testing.

In [ ]:
robust = rob[rob['n_pos_windows']>=2].sort_values('sharpe_mean', ascending=False)
joint_top = rob.sort_values('joint', ascending=False).head(5)
candidates = pd.concat([robust, joint_top]).drop_duplicates(subset=['cohort','bucket','sizing']).head(5)
candidates.style.format({
    'sharpe_mean':'{:.2f}','sharpe_min':'{:.2f}','sharpe_max':'{:.2f}',
    'ret_pct_total':'{:+.2%}','ret_pct_annual_mean':'{:+.2%}',
    'avg_deploy':'{:.0%}','avg_n_signals':'{:.0f}',
    'avg_fallback':'{:.0%}','joint':'{:.3f}',
})

---

**Caveats:**
- Low-signal-count runs (2d buckets have 37-45 signals across a year) carry wide Sharpe error bars. Treat as directional, not precise.
- 2024 sidebar runs have 33-86% fallback rates — most signals fell to the 3¢ adverse default because 2024 markets were thin. Don't draw conclusions from 2024 alone.
- The TopK score `mkt_total_pnl × mkt_profit_factor` is uncapped and dominated by traders with near-zero losses. Worth checking whether top-PnL runs are concentrated in 1-2 leaders with extreme PF.
- Style metrics for Cohort E (`style_role_balance`) use lifetime values from `traders.parquet` as approximation — small leak documented in METRICS_REFERENCE.